In [4]:
import importlib
from datetime import datetime

from vnpy.trader.constant import Interval
from vnpy_portfoliostrategy.backtesting import BacktestingEngine

# 如果策略代码有修改，需要reload
from vnpy_portfoliostrategy.my_strategies import risk_parity_etf_strategy
importlib.reload(risk_parity_etf_strategy)
from vnpy_portfoliostrategy.my_strategies.risk_parity_etf_strategy import EnhancedRiskParityStrategy

# 更丰富的ETF组合
SYMBOLS = [
    # 股票（3只）
    "510300.SSE",   # 沪深300
    "510500.SSE",   # 中证500
    "159915.SZSE",  # 创业板
    
    # 债券（2只）
    "511010.SSE",   # 国债ETF
    "511260.SSE",   # 10年国债
    
    # 黄金和海外（2只）
    "518880.SSE",   # 黄金ETF
    "513100.SSE",   # 纳指ETF
]

def get_default_config(symbols):
    config = {"rates": {}, "slippages": {}, "sizes": {}, "priceticks": {}}
    for symbol in symbols:
        config["rates"][symbol] = 3/10000      # 手续费万分之三
        config["slippages"][symbol] = 0.001    # 滑点
        config["sizes"][symbol] = 1            # ⚠️ 改为1，ETF一份就是1股
        config["priceticks"][symbol] = 0.001   # 最小价格变动
    return config

engine = BacktestingEngine()
config = get_default_config(SYMBOLS)

engine.set_parameters(
    vt_symbols=SYMBOLS,
    interval=Interval.DAILY,
    start=datetime(2019, 1, 1),
    end=datetime(2026, 5, 29),
    rates=config["rates"],
    slippages=config["slippages"],
    sizes=config["sizes"],
    priceticks=config["priceticks"],
    capital=1_000_000,
)

setting = {
    # 调仓
    "rebalance_interval": 21,        # 21天（月度），降低交易频率

    # 动态权重
    "use_dynamic_weights": True,
    "volatility_lookback": 60,       # 60天，更稳定
    "target_portfolio_vol": 0.15,    # 15%目标波动率

    # 市场状态
    "use_market_regime": True,
    "ma_trend_lookback": 200,

    # 风险平价（关闭，使用波动率倒数）
    "use_risk_parity": True,
    "risk_parity_lookback": 30,
    "risk_parity_max_iter": 1000,
    "risk_parity_tolerance": 1e-6,
    
    "bullish_equity_multiplier" : 1.2,  # 牛市中股票权重倍数
    "bullish_bond_multiplier" : 0.8,  # 牛市中债券权重倍数
    "bearish_equity_multiplier" : 0.7,  # 熊市中股票权重倍数
    "bearish_gold_multiplier" : 1.4,  # 熊市中黄金权重倍数
    "bearish_bond_multiplier" : 1.1,  # 熊市中债券权重倍数

    # 仓位管理
    "max_position_pct": 0.70,        # 最大总仓位70%
    "min_position_pct": 0.30,        # 最小总仓位30%
    "single_etf_max_pct": 0.35,      # 单只ETF最大35%
    "use_full_capital": True,        # ⚠️ 添加这个参数（True=动态现金管理）
    
    # 风险控制
    "max_drawdown_stop": 0.20,       # 最大回撤止损20%
    "trailing_stop": 0.12,           # 移动止损12%（十天之内下跌12%）
    "stop_loss_days": 20,            # 止损后冷却20天
    "use_dynamic_stop": False,        # 启用动态止损
    "peak_multiplier": 0.6,          # 创新高后止损收紧倍数（三十天内创新高后，到达回撤max_drawdown_stop * peak_multiplier就止损）
    
    # 再平衡
    "rebalance_threshold": 0.10,     # 权重偏离10%调仓
    
    # 基础参数
    "price_add": 0.01,               # ⚠️ 添加滑点补偿
    "initial_capital": 1_000_000,    # ⚠️ 添加初始资金
    "commission_rate": 0.0003,       # ⚠️ 添加手续费率
}

# 运行回测
engine.add_strategy(EnhancedRiskParityStrategy, setting)
engine.load_data()
engine.run_backtesting()
df = engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()

2026-06-12 11:04:09.353448	Start loading historical data
2026-06-12 11:04:09.354806	510300.SSE Historical data loading completed, data count: 1794
2026-06-12 11:04:09.355565	510500.SSE Historical data loading completed, data count: 1794
2026-06-12 11:04:09.356057	159915.SZSE Historical data loading completed, data count: 1793
2026-06-12 11:04:09.356579	511010.SSE Historical data loading completed, data count: 1794
2026-06-12 11:04:09.357208	511260.SSE Historical data loading completed, data count: 1794
2026-06-12 11:04:09.357737	518880.SSE Historical data loading completed, data count: 1794
2026-06-12 11:04:09.358304	513100.SSE Historical data loading completed, data count: 1793
2026-06-12 11:04:09.358311	All historical data has been loaded
2026-06-12 11:04:09.364750	Strategy initialization complete
2026-06-12 11:04:09.365099	Start replaying historical data
2026-06-12 11:04:09.597321	Historical backtest complete
2026-06-12 11:04:09.597400	Start calculating daily mark-to-market profit a

In [ ]:
print("\n" + "="*30 + " 策略日志输出 " + "="*30)
for log in engine.logs:
    print(log)  # 每条日志会包含时间和内容


============================== 策略日志输出 ==============================
1970-01-01 00:00:00	增强版风险平价策略初始化，初始资金: 1000000.00
1970-01-01 00:00:00	资金管理模式: 动态现金管理
2019-10-29 00:00:00+08:00	策略启动
2020-03-27 00:00:00+08:00	风险贡献出现非正值，使用简化版波动率倒数权重
2020-03-27 00:00:00+08:00	调仓执行: 510300.SSE: 0手 -> 6512手 (目标权重 3.9%); 510500.SSE: 0手 -> 16160手 (目标权重 3.6%); 159915.SZSE: 0手 -> 11504手 (目标权重 3.0%); 511010.SSE: 0手 -> 2129手 (目标权重 37.8%); 511260.SSE: 0手 -> 2348手 (目标权重 37.8%); 518880.SSE: 0手 -> 18590手 (目标权重 9.6%); 513100.SSE: 0手 -> 9520手 (目标权重 4.3%)
2020-03-27 00:00:00+08:00	调仓完成 | 权益: 1,000,000 | 现金: 1,000,000 | 趋势: bearish | 组合波动率: 6.6% 风险缩放: 1.00
2020-03-30 00:00:00+08:00	风险贡献出现非正值，使用简化版波动率倒数权重
2020-03-30 00:00:00+08:00	调仓执行: 510300.SSE: 6512手 -> 6576手 (目标权重 3.9%); 510500.SSE: 16160手 -> 16390手 (目标权重 3.6%); 159915.SZSE: 11504手 -> 11759手 (目标权重 3.0%); 511010.SSE: 2129手 -> 2127手 (目标权重 37.8%); 511260.SSE: 2348手 -> 2349手 (目标权重 37.8%); 518880.SSE: 18590手 -> 18694手 (目标权重 9.6%); 513100.SSE: 9520手 -> 9724手 (目标权重 4.3%

In [ ]:
if engine.trades:
    for trade in engine.trades.values():
        # 只打印 TradeData 实际有的字段
        print(f"时间: {trade.datetime} | 标的: {trade.symbol} | "
              f"方向: {trade.direction.value} | 开平: {trade.offset.value} | "
              f"价格: {trade.price:.3f} | 数量: {trade.volume}")
else:
    print("无成交记录")

时间: 2020-03-30 00:00:00+08:00 | 标的: 510300 | 方向: Long | 开平: Open | 价格: 4.138 | 数量: 6512
时间: 2020-03-30 00:00:00+08:00 | 标的: 510500 | 方向: Long | 开平: Open | 价格: 1.540 | 数量: 16160
时间: 2020-03-30 00:00:00+08:00 | 标的: 159915 | 方向: Long | 开平: Open | 价格: 1.807 | 数量: 11504
时间: 2020-03-30 00:00:00+08:00 | 标的: 511010 | 方向: Long | 开平: Open | 价格: 124.274 | 数量: 2129
时间: 2020-03-30 00:00:00+08:00 | 标的: 511260 | 方向: Long | 开平: Open | 价格: 112.650 | 数量: 2348
时间: 2020-03-30 00:00:00+08:00 | 标的: 518880 | 方向: Long | 开平: Open | 价格: 3.597 | 数量: 18590
时间: 2020-03-30 00:00:00+08:00 | 标的: 513100 | 方向: Long | 开平: Open | 价格: 3.066 | 数量: 9520
时间: 2020-03-31 00:00:00+08:00 | 标的: 510300 | 方向: Long | 开平: Open | 价格: 4.156 | 数量: 64
时间: 2020-03-31 00:00:00+08:00 | 标的: 510500 | 方向: Long | 开平: Open | 价格: 1.535 | 数量: 230
时间: 2020-03-31 00:00:00+08:00 | 标的: 159915 | 方向: Long | 开平: Open | 价格: 1.801 | 数量: 255
时间: 2020-03-31 00:00:00+08:00 | 标的: 511010 | 方向: Short | 开平: Close | 价格: 124.423 | 数量: 2
时间: 2020-03-31 00:00:00+08:0

In [ ]:
import importlib
from datetime import datetime
import pandas as pd
import os

from vnpy.trader.constant import Interval
from vnpy_portfoliostrategy.backtesting import BacktestingEngine

from vnpy_portfoliostrategy.my_strategies import risk_parity_etf_strategy
importlib.reload(risk_parity_etf_strategy)
from vnpy_portfoliostrategy.my_strategies.risk_parity_etf_strategy import EnhancedRiskParityStrategy

# 更丰富的ETF组合
SYMBOLS = [
    # 股票（3只）
    "510300.SSE",   # 沪深300
    "510500.SSE",   # 中证500
    "159915.SZSE",  # 创业板
    
    # 债券（2只）
    "511010.SSE",   # 国债ETF
    "511260.SSE",   # 10年国债
    
    # 黄金和海外（2只）
    "518880.SSE",   # 黄金ETF
    "513100.SSE",   # 纳指ETF
]

def get_default_config(symbols):
    config = {"rates": {}, "slippages": {}, "sizes": {}, "priceticks": {}}
    for symbol in symbols:
        config["rates"][symbol] = 3/10000
        config["slippages"][symbol] = 0.001
        config["sizes"][symbol] = 100
        config["priceticks"][symbol] = 0.001
    return config

# ========== 连续回测 ==========
print("开始连续回测（2015-2026）...")
print("="*60)

# 创建回测引擎
engine = BacktestingEngine()
config = get_default_config(SYMBOLS)

# 设置回测参数（包含足够的预热期）
engine.set_parameters(
    vt_symbols=SYMBOLS,
    interval=Interval.DAILY,
    start=datetime(2015, 1, 1),      # 提前2年加载数据，确保有足够预热
    end=datetime(2026, 5, 29),
    rates=config["rates"],
    slippages=config["slippages"],
    sizes=config["sizes"],
    priceticks=config["priceticks"],
    capital=1_000_000,
)

setting = {
    # 调仓
    "rebalance_interval": 21,        # 改为21天（月度），降低交易频率
    
    # 动态权重
    "use_dynamic_weights": True,
    "volatility_lookback": 60,       # 恢复60天，更稳定
    "target_portfolio_vol": 0.15,    # 改为15%目标波动率，更现实
    
    # 市场状态
    "use_market_regime": True,
    "ma_trend_lookback": 200,
    
    # 风险平价（关闭，使用波动率倒数）
    "use_risk_parity": False,
    "risk_parity_lookback": 60,
    "risk_parity_max_iter": 100,
    "risk_parity_tolerance": 1e-6,
    
    # 仓位管理（更保守）
    "max_position_pct": 0.70,        # 从0.9降到0.7
    "min_position_pct": 0.30,
    "single_etf_max_pct": 0.35,      # 从0.3提高到0.35（因为只有5只ETF）
    
    # 风险控制
    "max_drawdown_stop": 0.20,       # 放宽到20%，避免频繁止损
    "trailing_stop": 0.12,           # 放宽到12%
    "stop_loss_days": 20,
    "use_dynamic_stop": True,
    "peak_multiplier": 0.6,
    
    # 再平衡
    "rebalance_threshold": 0.10,
}

engine.add_strategy(EnhancedRiskParityStrategy, setting)
engine.load_data()
engine.run_backtesting()

# 获取每日数据
daily_df = engine.calculate_result()

# 确保索引是日期格式
if not isinstance(daily_df.index, pd.DatetimeIndex):
    daily_df.index = pd.to_datetime(daily_df.index)
daily_df = daily_df.sort_index()

# 计算净值
if 'balance' in daily_df.columns:
    daily_df['nav'] = daily_df['balance']
else:
    daily_df['nav'] = 1_000_000 + daily_df['net_pnl'].cumsum()

# 找到策略真正开始交易的日期（第一次有成交或净值开始变化）
strategy_start_date = None

# 查找第一次有成交的日期
if 'trade_count' in daily_df.columns:
    first_trade_df = daily_df[daily_df['trade_count'] > 0]
    if len(first_trade_df) > 0:
        strategy_start_date = first_trade_df.index[0]

# 如果没有成交，查找净值第一次变化的日期
if strategy_start_date is None:
    initial_nav = daily_df['nav'].iloc[0]
    nav_changed_df = daily_df[abs(daily_df['nav'] - initial_nav) > 1]
    if len(nav_changed_df) > 0:
        strategy_start_date = nav_changed_df.index[0]

print(f"\n策略预热完成，正式开始交易日期: {strategy_start_date.strftime('%Y-%m-%d')}")

# 从策略真正开始日期开始，过滤掉预热期
daily_df = daily_df[daily_df.index >= strategy_start_date]

# ========== 按年统计 ==========
print("\n" + "="*80)
print("📊 按年统计结果")
print("="*80)

# 添加年份列
daily_df['year'] = daily_df.index.year

annual_results = []

for year, group in daily_df.groupby('year'):
    if len(group) == 0:
        continue
    
    # 获取该年净值序列
    nav = group['nav']
    start_nav = nav.iloc[0]
    end_nav = nav.iloc[-1]
    
    # 起始日期和结束日期
    start_date = group.index[0].strftime('%Y-%m-%d')
    end_date = group.index[-1].strftime('%Y-%m-%d')
    
    # 总收益率
    total_return = (end_nav - start_nav) / start_nav
    
    # 年化收益（基于实际交易日数）
    trading_days = len(group)
    if trading_days > 0 and total_return > -1:
        annual_return = (1 + total_return) ** (252 / trading_days) - 1
    else:
        annual_return = 0
    
    # 最大回撤（使用全局累计最高点）
    cummax = nav.cummax()
    drawdown = (nav - cummax) / cummax
    max_drawdown = drawdown.min()
    
    # 夏普比率
    daily_returns = nav.pct_change().dropna()
    if len(daily_returns) > 0 and daily_returns.std() > 0:
        sharpe = (daily_returns.mean() * 252) / (daily_returns.std() * (252 ** 0.5))
    else:
        sharpe = 0
    
    # 成交统计
    total_trades = group['trade_count'].sum() if 'trade_count' in group.columns else 0
    
    annual_results.append({
        '年份': year,
        '起始日期': start_date,
        '结束日期': end_date,
        '年化收益': annual_return,
        '总收益': total_return,
        '最大回撤': max_drawdown,
        '夏普比率': sharpe,
        '起始净值': start_nav,
        '结束净值': end_nav,
        '交易日数': trading_days,
        '成交笔数': total_trades,
    })

# 创建显示用的DataFrame
display_df = pd.DataFrame([{
    '年份': r['年份'],
    '起始日期': r['起始日期'],
    '结束日期': r['结束日期'],
    '年化收益': f"{r['年化收益']:.2%}",
    '总收益': f"{r['总收益']:.2%}",
    '最大回撤': f"{r['最大回撤']:.2%}",
    '夏普比率': f"{r['夏普比率']:.2f}",
    '起始净值': f"{r['起始净值']:,.0f}",
    '结束净值': f"{r['结束净值']:,.0f}",
    '交易日数': r['交易日数'],
    '成交笔数': r['成交笔数']
} for r in annual_results])

print(display_df.to_string(index=False))

# ========== 整体统计 ==========
print(f"\n{'='*80}")
print("📈 整体统计")
print(f"{'='*80}")

if len(annual_results) > 0:
    # 整体收益（从策略开始到结束）
    total_nav_start = annual_results[0]['起始净值']
    total_nav_end = annual_results[-1]['结束净值']
    total_return_all = (total_nav_end - total_nav_start) / total_nav_start
    
    # 整体年化收益
    total_days = sum(r['交易日数'] for r in annual_results)
    if total_days > 0:
        total_annual_return = (1 + total_return_all) ** (252 / total_days) - 1
    else:
        total_annual_return = 0
    
    # 最大回撤（所有年份中最大的）
    max_drawdown_all = min(r['最大回撤'] for r in annual_results)
    
    # 平均夏普
    avg_sharpe = sum(r['夏普比率'] for r in annual_results) / len(annual_results)
    
    # 正收益年份
    positive_years = [r for r in annual_results if r['总收益'] > 0]
    
    print(f"策略开始日期: {annual_results[0]['起始日期']}")
    print(f"策略结束日期: {annual_results[-1]['结束日期']}")
    print(f"整体总收益: {total_return_all:.2%}")
    print(f"整体年化收益: {total_annual_return:.2%}")
    print(f"最大回撤: {max_drawdown_all:.2%}")
    print(f"平均夏普比率: {avg_sharpe:.2f}")
    print(f"正收益年份: {len(positive_years)}/{len(annual_results)} ({len(positive_years)/len(annual_results):.0%})")
    print(f"总成交笔数: {sum(r['成交笔数'] for r in annual_results)}")

# 保存到CSV
desktop = os.path.join(os.path.expanduser('~'), 'Desktop')
file_path = os.path.join(desktop, 'Annual_backtest_results.csv')

save_df = pd.DataFrame([{
    '年份': r['年份'],
    '起始日期': r['起始日期'],
    '结束日期': r['结束日期'],
    '年化收益_%': f"{r['年化收益']:.2%}",
    '总收益_%': f"{r['总收益']:.2%}",
    '最大回撤_%': f"{r['最大回撤']:.2%}",
    '夏普比率': f"{r['夏普比率']:.2f}",
    '起始净值': r['起始净值'],
    '结束净值': r['结束净值'],
    '交易日数': r['交易日数'],
    '成交笔数': r['成交笔数']
} for r in annual_results])

save_df.to_csv(file_path, index=False, encoding='utf-8-sig')
print(f"\n✅ 结果已保存到: {file_path}")


开始连续回测（2015-2026）...
2026-06-11 13:26:56.718521	Start loading historical data
2026-06-11 13:26:56.720727	510300.SSE Historical data loading completed, data count: 2769
2026-06-11 13:26:56.721801	510500.SSE Historical data loading completed, data count: 2767
2026-06-11 13:26:56.723515	159915.SZSE Historical data loading completed, data count: 2768
2026-06-11 13:26:56.724466	511010.SSE Historical data loading completed, data count: 2769
2026-06-11 13:26:56.725701	511260.SSE Historical data loading completed, data count: 2123
2026-06-11 13:26:56.726713	518880.SSE Historical data loading completed, data count: 2769
2026-06-11 13:26:56.727676	513100.SSE Historical data loading completed, data count: 2768
2026-06-11 13:26:56.727684	All historical data has been loaded
2026-06-11 13:26:56.734554	Strategy initialization complete
2026-06-11 13:26:56.734618	Start replaying historical data
2026-06-11 13:26:57.068201	Historical backtest complete
2026-06-11 13:26:57.068344	Start calculating daily ma

In [2]:
import pandas as pd

# 创建包含balance的DataFrame
balance_df = pd.DataFrame({
    'date': df.index,
    'balance': df['balance'].values
})

# 导出为xlsx
balance_df.to_excel('strategy_balance.xlsx', index=False)
print("✅ 已导出 strategy_balance.xlsx")
print(f"数据量: {len(balance_df)} 行")
print(f"初始资金: {balance_df['balance'].iloc[0]:.2f}")
print(f"最终资金: {balance_df['balance'].iloc[-1]:.2f}")

✅ 已导出 strategy_balance.xlsx
数据量: 1595 行
初始资金: 1000000.00
最终资金: 1530627.48


In [3]:
import pandas as pd

initial_capital = 100_0000  # 100万

# 1. 读 close_price
price_df = pd.read_excel("close_price.xlsx")

# 假设列是 close_price + date（或 index 是 date）
if "date" in price_df.columns:
    price_df["date"] = pd.to_datetime(price_df["date"])
    price_df = price_df.set_index("date")
else:
    price_df.index = pd.to_datetime(price_df.index)

# 2. 确保 df 也有 datetime index
df.index = pd.to_datetime(df.index)

# 3. 对齐时间（关键步骤）
merged = pd.DataFrame(index=df.index).join([
    df["balance"],
    price_df["close_price"]
], how="inner")

# 4. 计算 buy & hold NAV
merged["buy_hold"] = (
    merged["close_price"] / merged["close_price"].iloc[0]
) * initial_capital

# 5. 导出 Excel
merged.to_excel("backtest_result.xlsx")